# 🚀 Linear Block Diffusion — Kaggle Training & Evaluation

**Architecture**: `LinearBlockDiffusionArchitecture` from `pymbbo`  
**Hardware**: Kaggle Dual NVIDIA Tesla T4 GPUs (DataParallel + FP16 AMP)  
**Tokenizer**: `GPT2TokenizerFast` (vocab = 50 257)  
**Dataset**: `wikitext-2-raw-v1` de HuggingFace (bloques de 1024 tokens)  

### Fixes aplicados
- ✅ `std=0.02` init → loss inicial ~10.8
- ✅ `return_logits=False` → sin bottleneck PCIe
- ✅ Chunked stable scan (32 iters Python en vez de 1024)
- ✅ Loss SOLO sobre posiciones enmascaradas → convergencia monotónica
- ✅ Scan siempre en **float32** interno → sin NaN por overflow FP16

In [ ]:
# ── Cell 1: Instalar pymbbo SIN tocar el entorno de Kaggle ──────────────
# --no-deps es CRÍTICO: impide que pip reinstale numpy/scipy/torch
!pip install -q --no-deps --force-reinstall git+https://github.com/bueormnew/pymbbo.git
print('✅ pymbbo instalado')

In [ ]:
# ── Cell 2: Hardware ────────────────────────────────────────────────────
import os, sys, math, time
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print('=' * 70)
print('🖥️  HARDWARE DIAGNOSTIC')
print('=' * 70)
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')

NUM_GPUS = 0
if torch.cuda.is_available():
    NUM_GPUS = torch.cuda.device_count()
    for i in range(NUM_GPUS):
        p = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}] {p.name} | {p.total_memory/1e9:.1f} GB')
else:
    print('⚠️  Sin GPU — activa en Kaggle Settings > Accelerator')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Cell 3: Patch typing + import pymbbo ───────────────────────────────
import typing, builtins
for _n in ['Tuple','List','Dict','Optional','Union','Any','Callable']:
    if not hasattr(builtins, _n):
        setattr(builtins, _n, getattr(typing, _n))

from pymbbo.architectures.linear_block_diffusion import LinearBlockDiffusionArchitecture
print('✅ LinearBlockDiffusionArchitecture importada')

In [ ]:
# ── Cell 4: Dataset + GPT-2 Tokenizer ─────────────────────────────────
# GPT2TokenizerFast: no arrastra AutoTokenizer → sin sklearn/scipy import
from datasets import load_dataset
from transformers import GPT2TokenizerFast

print('=' * 70)
print('🤗 TOKENIZER & DATASET')
print('=' * 70)

tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = len(tokenizer)   # 50257
print(f'GPT-2 tokenizer: vocab={VOCAB_SIZE:,}')

raw_dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
train_texts = [t for t in raw_dataset['train']['text']      if len(t.strip()) > 50]
val_texts   = [t for t in raw_dataset['validation']['text'] if len(t.strip()) > 50]
print(f'Textos: Train={len(train_texts):,} | Val={len(val_texts):,}')

SEQ_LEN, PROMPT_LEN = 1024, 64
TOTAL_LEN = PROMPT_LEN + SEQ_LEN

def tokenize_and_chunk(texts, max_samples=8000):
    all_ids = []
    for text in texts:
        all_ids.extend(tokenizer.encode(text))
        if len(all_ids) >= max_samples * 512:
            break
    prompts, targets = [], []
    for i in range(0, len(all_ids) - TOTAL_LEN, 256):
        prompts.append(all_ids[i : i + PROMPT_LEN])
        targets.append(all_ids[i + PROMPT_LEN : i + TOTAL_LEN])
        if len(prompts) >= max_samples:
            break
    return torch.tensor(prompts, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

train_p, train_t = tokenize_and_chunk(train_texts, max_samples=8000)
val_p,   val_t   = tokenize_and_chunk(val_texts,   max_samples=1000)

class TextPairDataset(Dataset):
    def __init__(self, p, t): self.p, self.t = p, t
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.p[i], self.t[i]

BATCH_SIZE   = 8
train_loader = DataLoader(TextPairDataset(train_p, train_t), batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(TextPairDataset(val_p, val_t), batch_size=BATCH_SIZE,
                          shuffle=False, drop_last=True, num_workers=2, pin_memory=True)
print(f'Train={len(train_p):,} | Val={len(val_p):,} | Batch={BATCH_SIZE}')

In [ ]:
# ── Cell 5: Modelo ────────────────────────────────────────────────────
print('=' * 70)
print('🏗️  MODELO LinearBlockDiffusion')
print('=' * 70)

D_MODEL             = 512
NUM_LAYERS          = 6
BLOCK_SIZE          = 512
OVERLAP_RATIO       = 0.5
NUM_DIFFUSION_STEPS = 8
CHUNK_DENOISE_SIZE  = 64

raw_model = LinearBlockDiffusionArchitecture(
    vocab_size          = VOCAB_SIZE,
    d_model             = D_MODEL,
    num_layers          = NUM_LAYERS,
    block_size          = BLOCK_SIZE,
    overlap_ratio       = OVERLAP_RATIO,
    num_diffusion_steps = NUM_DIFFUSION_STEPS,
    chunk_denoise_size  = CHUNK_DENOISE_SIZE,
    pad_token_id        = tokenizer.eos_token_id,
    eos_token_id        = tokenizer.eos_token_id,
    noise_injection_prob = 0.15,
    dropout             = 0.1,
)

num_params = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
print(f'Parámetros: {num_params/1e6:.2f}M')

raw_model = raw_model.to(DEVICE)
if NUM_GPUS > 1:
    print(f'🔗 DataParallel: {NUM_GPUS} GPUs')
    model = nn.DataParallel(raw_model)
else:
    model = raw_model

# Warmup forward para que el scan kernel se estabilice antes del entrenamiento
with torch.no_grad():
    _p = torch.randint(0, 100, (1, 8)).to(DEVICE)
    _t = torch.randint(0, 100, (1, 32)).to(DEVICE)
    raw_model(_p, target_ids=_t, return_logits=False)
    del _p, _t
if torch.cuda.is_available(): torch.cuda.synchronize()
print(f'✅ Modelo listo en {DEVICE}')

In [ ]:
# ── Cell 6: Entrenamiento ─────────────────────────────────────────────
print('=' * 70)
print('⚡ ENTRENAMIENTO — TOKENS/S, VRAM, MS/PASO')
print('=' * 70)

EPOCHS              = 3
LR                  = 3e-4      # Más conservador que 5e-4
WARMUP_STEPS        = 50        # Calentamiento lineal del LR
MAX_STEPS_PER_EPOCH = 200
NAN_PATIENCE        = 5         # Pasos NaN consecutivos antes de abortar

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01, betas=(0.9, 0.95))
total_steps  = EPOCHS * min(MAX_STEPS_PER_EPOCH, len(train_loader))
scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)
scaler       = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

total_tokens_trained = 0
total_train_time     = 0.0
step_count           = 0
global_step          = 0
train_losses         = []
peak_vram_train      = 0.0
nan_streak           = 0
aborted              = False

model.train()
for epoch in range(1, EPOCHS + 1):
    if aborted: break
    epoch_loss, epoch_steps = 0.0, 0
    print(f'\n{"─"*70}  EPOCH {epoch}/{EPOCHS}')

    for step, (p_batch, t_batch) in enumerate(train_loader):
        if step >= MAX_STEPS_PER_EPOCH or aborted: break

        # Warmup lineal del LR
        if global_step < WARMUP_STEPS:
            warmup_factor = (global_step + 1) / WARMUP_STEPS
            for pg in optimizer.param_groups:
                pg['lr'] = LR * warmup_factor

        t0 = time.perf_counter()
        p_batch = p_batch.to(DEVICE, non_blocking=True)
        t_batch = t_batch.to(DEVICE, non_blocking=True)
        batch_tokens = t_batch.numel()

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            loss = model(p_batch, target_ids=t_batch, return_logits=False)
            if isinstance(model, nn.DataParallel):
                loss = loss.mean()

        # NaN detection: si el forward ya da NaN, no propagar
        if not torch.isfinite(loss):
            nan_streak += 1
            print(f'  ⚠️  NaN en paso {step+1} (racha: {nan_streak}/{NAN_PATIENCE}) — saltando')
            if nan_streak >= NAN_PATIENCE:
                print('  🛑  Demasiados NaN consecutivos — abortando entrenamiento')
                aborted = True
            optimizer.zero_grad(set_to_none=True)
            continue
        nan_streak = 0

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Si hay NaN en gradientes, skipear el paso del optimizer
        if not torch.isfinite(grad_norm):
            print(f'  ⚠️  Gradiente NaN/Inf en paso {step+1} — saltando optimizer')
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.step(optimizer)
        scaler.update()
        if global_step >= WARMUP_STEPS:
            scheduler.step()

        if torch.cuda.is_available(): torch.cuda.synchronize()
        dt = time.perf_counter() - t0

        total_train_time     += dt
        total_tokens_trained += batch_tokens
        step_count  += 1
        global_step += 1
        l = loss.item()
        epoch_loss  += l
        epoch_steps += 1
        train_losses.append(l)

        if torch.cuda.is_available():
            peak_vram_train = max(peak_vram_train, torch.cuda.max_memory_allocated()/1e9)

        if (step + 1) % 25 == 0 or step == 0:
            ppl = math.exp(min(l, 20.0))
            cur_lr = optimizer.param_groups[0]['lr']
            print(f'  [{step+1:3d}/{MAX_STEPS_PER_EPOCH}] '
                  f'Loss: {l:.4f} | PPL: {ppl:8.2f} | '
                  f'{dt*1000:6.1f} ms/paso | {batch_tokens/dt:>7,.0f} tok/s | '
                  f'VRAM: {peak_vram_train:.2f} GB | LR: {cur_lr:.2e}')

    if epoch_steps > 0:
        avg = epoch_loss / epoch_steps
        print(f'  → Epoch {epoch} avg: Loss={avg:.4f} | PPL={math.exp(min(avg,20)):.2f}')

if aborted:
    print('\n🛑 Entrenamiento abortado por NaN — revisa la versión de pymbbo instalada.')
else:
    avg_tok_s       = total_tokens_trained / total_train_time
    avg_ms_per_step = (total_train_time / step_count) * 1000
    print(f'\n{"="*70}')
    print(f'✅ {step_count} pasos | {avg_tok_s:,.0f} tok/s | {avg_ms_per_step:.1f} ms/paso | VRAM: {peak_vram_train:.2f} GB')

In [ ]:
# ── Cell 7: Evaluación Validación ──────────────────────────────────────
print('=' * 70)
print('📉 EVALUACIÓN — VALIDATION LOSS & PPL')
print('=' * 70)

model.eval()
val_loss_sum, val_steps = 0.0, 0
MAX_VAL_STEPS = 50

with torch.no_grad():
    for p_batch, t_batch in val_loader:
        if val_steps >= MAX_VAL_STEPS: break
        p_batch = p_batch.to(DEVICE, non_blocking=True)
        t_batch = t_batch.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            loss = model(p_batch, target_ids=t_batch, return_logits=False)
            if isinstance(model, nn.DataParallel):
                loss = loss.mean()
        if torch.isfinite(loss):
            val_loss_sum += loss.item()
            val_steps    += 1

avg_val_loss = val_loss_sum / max(val_steps, 1)
val_ppl      = math.exp(min(avg_val_loss, 20.0))
print(f'  Validation Loss : {avg_val_loss:.4f}')
print(f'  Validation PPL  : {val_ppl:.2f}')
print(f'  Pasos evaluados : {val_steps}')

In [ ]:
# ── Cell 8: Benchmark Inferencia ───────────────────────────────────────
print('=' * 70)
print('🎯 INFERENCIA — BENCHMARK & GENERACIÓN')
print('=' * 70)

eval_model = raw_model
eval_model.eval()

test_prompts = [
    'In a distant world, scientists discovered',
    'The history of artificial intelligence began',
    'Once upon a time in a small village',
]

MAX_NEW_TOKENS = 256
all_infer_times, all_infer_tokens = [], []

if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

for i, prompt_text in enumerate(test_prompts):
    prompt_ids = tokenizer.encode(prompt_text, return_tensors='pt').to(DEVICE)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()

    generated_ids = eval_model.generate(
        prompt_ids, max_new_tokens=MAX_NEW_TOKENS,
        block_size=512, overlap_ratio=0.5,
        num_diffusion_steps=8, chunk_denoise_size=64,
        temperature=0.8, top_k=40,
        eos_token_id=None,
    )

    if torch.cuda.is_available(): torch.cuda.synchronize()
    dt = time.perf_counter() - t0

    gen_count = generated_ids.shape[1] - prompt_ids.shape[1]
    all_infer_times.append(dt)
    all_infer_tokens.append(gen_count)
    decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f'\n{"─"*70}')
    print(f'  Sample {i+1} | "{prompt_text}"')
    print(f'  {gen_count} tokens | {dt:.2f}s | {gen_count/dt:.1f} tok/s')
    print(f'{"─"*70}')
    print(decoded[:600])

total_infer_tok  = sum(all_infer_tokens)
total_infer_time = sum(all_infer_times)
avg_infer_tok_s  = total_infer_tok / total_infer_time
total_blocks     = sum(math.ceil(t / 256) for t in all_infer_tokens)
avg_ms_per_block = (total_infer_time / total_blocks) * 1000
infer_peak_vram  = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0.0
print(f'\n{"="*70}')
print(f'📊 {avg_infer_tok_s:.1f} tok/s | {avg_ms_per_block:.1f} ms/bloque | VRAM: {infer_peak_vram:.2f} GB')

In [ ]:
# ── Cell 9: Curvas ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

if len(train_losses) < 2:
    print('⚠️  Sin datos suficientes para graficar.')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    W = min(20, len(train_losses) // 4 or 1)

    ax1.plot(train_losses, lw=0.8, alpha=0.4, label='Loss por paso')
    if len(train_losses) >= W:
        ma = [sum(train_losses[i:i+W])/W for i in range(len(train_losses)-W+1)]
        ax1.plot(range(W-1, len(train_losses)), ma, lw=2, color='red', label=f'MA-{W}')
    ax1.axhline(avg_val_loss, color='orange', ls='--', lw=1.5, label=f'Val={avg_val_loss:.3f}')
    ax1.set(xlabel='Paso', ylabel='Loss', title='📉 Training Loss')
    ax1.legend(); ax1.grid(alpha=0.3)

    ppls = [math.exp(min(l, 15)) for l in train_losses]
    ax2.plot(ppls, lw=0.8, alpha=0.4, color='green', label='PPL por paso')
    if len(ppls) >= W:
        ma_p = [sum(ppls[i:i+W])/W for i in range(len(ppls)-W+1)]
        ax2.plot(range(W-1, len(ppls)), ma_p, lw=2, color='darkgreen', label=f'MA-{W}')
    ax2.axhline(val_ppl, color='orange', ls='--', lw=1.5, label=f'Val PPL={val_ppl:.1f}')
    ax2.set(xlabel='Paso', ylabel='Perplejidad', title='📊 Training PPL')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.suptitle('LinearBlockDiffusion — Curvas de Entrenamiento', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
    print('✅ Curvas generadas.')

In [ ]:
# ── Cell 10: Reporte Final ─────────────────────────────────────────────
print('=' * 70)
print('🏆 REPORTE FINAL')
print('=' * 70)

df_config = pd.DataFrame({
    'Parámetro': ['Params','d_model','Capas','Vocab','Block','K steps','Épocas','Batch','LR'],
    'Valor':     [f'{num_params/1e6:.2f}M', str(D_MODEL), str(NUM_LAYERS),
                  f'{VOCAB_SIZE:,}', str(BLOCK_SIZE), str(NUM_DIFFUSION_STEPS),
                  str(EPOCHS), str(BATCH_SIZE), str(LR)]
})
display(df_config)

_last_loss = train_losses[-1] if train_losses else float('nan')
df_perf = pd.DataFrame({
    'Métrica': [
        '⚡ Train tok/s','⚡ ms/paso','💾 Peak VRAM (train)',
        '📉 Loss final','📉 PPL final',
        '✅ Val Loss','✅ Val PPL',
        '🚀 Infer tok/s','🚀 ms/bloque','💾 Peak VRAM (infer)',
    ],
    'Valor': [
        f'{avg_tok_s:,.0f} tok/s' if not aborted else 'N/A',
        f'{avg_ms_per_step:.1f} ms' if not aborted else 'N/A',
        f'{peak_vram_train:.2f} GB',
        f'{_last_loss:.4f}', f'{math.exp(min(_last_loss,20)):.2f}',
        f'{avg_val_loss:.4f}', f'{val_ppl:.2f}',
        f'{avg_infer_tok_s:.1f} tok/s', f'{avg_ms_per_block:.1f} ms',
        f'{infer_peak_vram:.2f} GB',
    ]
})
display(df_perf)

df_samples = pd.DataFrame({
    'Muestra': [f'Sample {i+1}' for i in range(len(test_prompts))],
    'Prompt':  [p[:40]+'...' for p in test_prompts],
    'Tokens':  all_infer_tokens,
    'Tiempo':  [f'{t:.2f}s' for t in all_infer_times],
    'tok/s':   [f'{t/d:.1f}' for t,d in zip(all_infer_tokens, all_infer_times)],
})
display(df_samples)

print('✅ Benchmark completado.')